# 02 - Preprocessing

## Objective
Clean and prepare the real Daraz Nepal reviews for feature extraction.

## Input
- Raw scraped reviews: `data/raw/scraped_reviews/scraped_reviews.csv`

## Steps
1. Load raw data
2. Drop Devanagari script reviews (keep Romanized only)
3. Clean text (lowercase, remove punctuation, URLs)
4. Drop empty/very short reviews after cleaning
5. Split into train, val, test
6. Save cleaned splits to `data/processed/`

In [2]:
import pandas as pd
import re
import os
from sklearn.model_selection import train_test_split

# Load raw data
df = pd.read_csv('../data/raw/scraped_reviews/scraped_reviews.csv')
print("Raw data shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nClass distribution:")
print(df['sentiment_label'].value_counts())

Raw data shape: (4429, 6)
Columns: ['itemId', 'rating', 'review_text', 'review_date', 'source', 'sentiment_label']

Class distribution:
sentiment_label
positive    2019
neutral     1210
negative    1200
Name: count, dtype: int64


In [3]:
# Step 1 - Drop Devanagari script reviews
df = df[~df['review_text'].str.contains(r'[\u0900-\u097F]', regex=True)]
print("After dropping Devanagari:", len(df))

# Step 2 — Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['review_text'].apply(clean_text)

# Step 3 - Drop empty/very short reviews
df = df[df['cleaned_text'].str.strip().str.len() > 2]
print("After dropping short reviews:", len(df))
print("\nFinal class distribution:")
print(df['sentiment_label'].value_counts())

After dropping Devanagari: 4355
After dropping short reviews: 4343

Final class distribution:
sentiment_label
positive    1985
neutral     1185
negative    1173
Name: count, dtype: int64


In [4]:
# Step 4 - Split into train, val, test
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df['sentiment_label'])
train, val = train_test_split(train_val, test_size=0.176, random_state=42, stratify=train_val['sentiment_label'])

print("Train:", len(train))
print("Val:  ", len(val))
print("Test: ", len(test))

print("\nTrain distribution:")
print(train['sentiment_label'].value_counts())
print("\nVal distribution:")
print(val['sentiment_label'].value_counts())
print("\nTest distribution:")
print(test['sentiment_label'].value_counts())

Train: 3041
Val:   650
Test:  652

Train distribution:
sentiment_label
positive    1390
neutral      830
negative     821
Name: count, dtype: int64

Val distribution:
sentiment_label
positive    297
neutral     177
negative    176
Name: count, dtype: int64

Test distribution:
sentiment_label
positive    298
neutral     178
negative    176
Name: count, dtype: int64


In [5]:
# Step 5 - Save cleaned splits
os.makedirs('../data/processed', exist_ok=True)

train.to_csv('../data/processed/train_cleaned.csv', index=False)
val.to_csv('../data/processed/val_cleaned.csv', index=False)
test.to_csv('../data/processed/test_cleaned.csv', index=False)

print("Saved train_cleaned.csv:", len(train), "rows")
print("Saved val_cleaned.csv:  ", len(val), "rows")
print("Saved test_cleaned.csv: ", len(test), "rows")

Saved train_cleaned.csv: 3041 rows
Saved val_cleaned.csv:   650 rows
Saved test_cleaned.csv:  652 rows
